# Session 1: PydanticAI 101
We look at the core features of PydanticAI:

- `Agent` - the primary container for functionality. Unlike other frameworks that separate "chains" or "graphs," the Agent holds the model configuration, the system prompt logic, and the tools.

In [ ]:
import os

import logfire
from pydantic_ai import Agent, RunContext
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    openai_api_key: str
    open_ai_default_model: str = "openai:gpt-5-nano"


settings = Settings()

## Part 1: Hello World
How do you execute the most basic tasks of calling an LLM using Pydantic? One important aspect hidden from us here is authentication against the OpenAI API, which happens using a key we've set-up in our .env file.

In [ ]:
# Define the agent
# Note: You can swap 'openai:gpt-5-nano' for 'anthropic:claude-3-5-sonnet-latest' or 'gemini-1.5-pro'
simple_agent = Agent(
    settings.open_ai_default_model,
    system_prompt="You are a helpful, concise AI assistant. Answer in one sentence.",
)

# Run the agent (Jupyter supports top-level await)
result = await simple_agent.run("What is the capital of France?")

print(f"Response: {result.output}")
print(f"Usage: {result.usage()}")

### System Prompts: Static vs. Dynamic

**What is a system prompt?**  
The system prompt is the "background instructions" given to the LLM before it sees the user's input. It sets the AI's persona, constraints, and behavior.

**Static System Prompts**  
The simplest approach — a fixed string defined at agent creation:
```python
agent = Agent(..., system_prompt="You are a helpful assistant.")
```
This works when your instructions never change.

**Dynamic System Prompts**  
Sometimes you need your system prompt to adapt based on context — the current user, time of day, feature flags, or data fetched at runtime. PydanticAI handles this with the `@agent.system_prompt` decorator:

```python
@agent.system_prompt
def get_system_prompt() -> str:
    return f"Today is {date.today()}. Be helpful."
```

**Why use a decorator instead of f-strings?**
- **Lazy evaluation**: The function runs *when the agent is called*, not when it's defined. This matters when context changes between calls.
- **Separation of concerns**: Prompt logic lives in a dedicated function, not buried in agent configuration.
- **Testability**: You can unit test your prompt-building logic independently.

In [ ]:
from datetime import datetime

dynamic_agent = Agent(
    settings.open_ai_default_model,
    deps_type=str,  # We define that we expect a string dependency (e.g., user name)
)


@dynamic_agent.system_prompt
def add_time_and_user(ctx: RunContext[str]) -> str:
    # ctx.deps contains whatever we pass into 'deps' during .run()
    user_name = ctx.deps
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")

    return f"""
    The current time is {current_time}.
    You are speaking to {user_name}.
    Be polite but informal.
    """


# Run with dependencies
result = await dynamic_agent.run("What time is it? Who are you talking to?", deps="Alice")
print(result.output)

## Part 2: Conversation History (Memory)
PydanticAI does not hide memory inside a "MemoryChain" object. State is explicit. When you run an agent, you get back a RunResult object which contains new_messages(). To continue a conversation, you simply pass these messages back into the next run.

Beware long conversations - they can eventually exceed your context window length. Also, they can cost a few bob!

In [ ]:
# 1. Initialize the conversation
chat_agent = Agent(settings.open_ai_default_model, system_prompt="You are a sarcastic robot.")
messages = []  # Store history here

# 2. First Turn
print("--- Turn 1 ---")
result1 = await chat_agent.run("My name is Dave.", message_history=messages)
print("User: My name is Dave.")
print(f"Bot: {result1.output}")

# Update history
# result1.new_messages() returns the User prompt AND the Model response
messages += result1.new_messages()

# 3. Second Turn (Agent remembers the name)
print("\n--- Turn 2 ---")
result2 = await chat_agent.run("What is my name?", message_history=messages)
print("User: What is my name?")
print(f"Bot: {result2.output}")

While messages are just a list of objects - they're not just a list of strings but complete Pydantic objects, ModelRequest (for the user inputs) and ModelResponse (as the name implies - the responses).

In [ ]:
from rich import print as rprint  # Alex - TIL you could do this!

# Let's inspect the messages list to verify it's just a list of objects
rprint(messages)

## Part 3: Synchronous vs Asynchronous Execution
PydanticAI supports both synchronous (`run_sync()`) and asynchronous (`run()`) execution. However, the "convenience" wrapper used for agent.run_sync() does use run_until_complete() under the hood 🫠 which makes it try to create its own event loop (kinda ironic for a synchronous function).
From PydanticAI's documentation:

> Synchronously run the agent with a user prompt. This is a convenience method that wraps self.run with loop.run_until_complete(...). You therefore can't use this method inside async code or if there's an active event loop.

**Key Differences:**
- **Synchronous (`run_sync()`)**: Blocks execution until the agent completes. Simple to use, but inefficient when making multiple calls. Not possible in Jupyter notebooks.
- **Asynchronous (`await run()`)**: Non-blocking, allows concurrent execution. Essential for production applications that need to handle multiple requests or batch operations.

Here's an example of running multiple agent calls serially and then asynchronously using asyncio to manage them.

In [ ]:
import time

# Create an agent for our examples
agent = Agent(
    settings.open_ai_default_model,
    system_prompt="You are a helpful assistant. Answer concisely in one sentence.",
)

# Synchronous execution: Sequential (one after another)
print("=== Synchronous Execution (Sequential) ===")
start_time = time.time()

result1 = await agent.run("What is Python?")
print(f"1. {result1.output}")

result2 = await agent.run("What is JavaScript?")
print(f"2. {result2.output}")

result3 = await agent.run("What is Rust?")
print(f"3. {result3.output}")

sync_time = time.time() - start_time
print(f"\nTotal time (synchronous): {sync_time:.2f} seconds")

In [ ]:
import asyncio

# Asynchronous execution: Concurrent (all at once)
print("=== Asynchronous Execution (Concurrent) ===")
start_time = time.time()

# Run all three agents concurrently
results = await asyncio.gather(
    agent.run("What is Python?"),
    agent.run("What is JavaScript?"),
    agent.run("What is Rust?"),
)

for i, result in enumerate(results, 1):
    print(f"{i}. {result.output}")

async_time = time.time() - start_time
print(f"\nTotal time (asynchronous): {async_time:.2f} seconds")
print(f"Speedup: {sync_time / async_time:.2f}x faster!")

## 🧠 Practical Exercise: "The Tech Lead Simulator"
**Goal:**  
We'd like you to combine Dynamic System Prompts and Run Context to build a role-playing agent.

**Specification:**  
Create a tech_lead_agent that can answer questions at the "appropriate" level based on the experience and role of the user.

Use the existing dataclass UserContext containing role (str) and experience_level (str).

Use a dynamic system prompt to change the agent's persona:
* If role is "Junior", explain things simply and encourage them.
* If role is "Senior", be terse and technical.
* If experience_level is "Expert", assume they know the acronyms, otherwise explain them.

In [ ]:
from dataclasses import dataclass


# 1. Define your Dependency Schema - yes Pydantic works just fine with native dataclasses
@dataclass
class UserContext:
    role: str
    experience_level: str


# 2. Initialize Agent - note that you define a type "UserContext" for the dependencies
tech_lead_agent = Agent(settings.open_ai_default_model, deps_type=UserContext)


# 3. Define Dynamic Prompt Logic
@tech_lead_agent.system_prompt
def set_persona(ctx: RunContext[UserContext]) -> str:
    if ctx.deps.role == "Junior":
        return "You are a supportive mentor. Explain concepts simply using analogies."
    elif ctx.deps.role == "Senior":
        return "You are a busy CTO. Be extremely concise. Use technical jargon."
    return "You are a helpful assistant."


# 4. Test Cases
junior_dev = UserContext(role="Junior", experience_level="Novice")
senior_dev = UserContext(role="Senior", experience_level="Expert")

# Usage
res_jr = await tech_lead_agent.run("Why K8s?", deps=junior_dev)
print(f"To Junior: {res_jr.output}\n")

res_sr = await tech_lead_agent.run("Why K8s?", deps=senior_dev)
print(f"To Senior: {res_sr.output}")